# ALCF through the AmSC RIG

This notebook uses the **experimental** RIG helper in `amsc-client==0.7.0`.

```text
AMSC_TOKEN (AmSC Keycard/PAT) -> RIG -> facility IRI API
```

This is distinct from direct `client.facility(...)`, which uses facility-native authentication. RIG may still require an active facility allocation and configured Credential Vault or token-exchange access. Never paste, print, decode, compare, serialize, or commit a token.

**Validation status (2026-09-28):** this notebook is **statically validated** against the published wheel. The protected reads and optional workflow are **not live-validated** by this repository campaign. Submission is disabled by default.

This example discovers ALCF, proves the routed credential with a protected project read, inspects Polaris/Home, and provides an optional bounded job workflow.

In [ ]:
import os
from importlib.metadata import version
from amsc_client import Client
from amsc_client.core.exceptions import ApiError, AuthenticationError

STAGING_API = "https://api.staging.american-science-cloud.org/api/current"
STAGING_RIG = "https://rig.staging.american-science-cloud.org"
AMSC_TOKEN = os.environ["AMSC_TOKEN"]
assert version("amsc-client") == "0.7.0", "Install requirements.txt and restart this kernel"
client = Client(token=AMSC_TOKEN, base_url=STAGING_API)
print("Client release:", version("amsc-client"))

## Discover and connect
`rig_facilities()` uses the AmSC Keycard/PAT. The subsequent `projects()` call is protected and therefore stronger authentication evidence than public resource discovery.

In [ ]:
facilities = client.rig_facilities(rig_url=STAGING_RIG)
alcf_meta = next((f for f in facilities if f.name.lower() == "alcf"), None)
if alcf_meta is None:
    raise RuntimeError("ALCF is not advertised by this RIG deployment")
print(alcf_meta.name, alcf_meta.display_name, alcf_meta.tier, alcf_meta.api_version)
alcf = client.facility_via_rig("alcf", rig_url=STAGING_RIG)

## Protected read, then resource discovery
A successful project read proves the routed credential. An empty project list is possible for an authenticated user without visible projects.

In [ ]:
projects = alcf.projects()
print(f"Protected project read succeeded ({len(projects)} visible project(s))")
resources = alcf.resources()
for resource in resources:
    print(resource.name, resource.resource_type, resource.status)
polaris = alcf.resource("Polaris")
home = alcf.resource("Home")

## Optional Polaris submission (default off)
Set the environment values before enabling the gate. Output stays in the existing Home root; run-unique names prevent collisions. Enabling this cell creates a real scheduler job.

In [ ]:
from datetime import datetime, timezone

SUBMIT_JOB = False
ALCF_USERNAME = os.getenv("ALCF_USERNAME", "")
ALCF_ACCOUNT = os.getenv("ALCF_ACCOUNT", "")
ALCF_QUEUE = os.getenv("ALCF_QUEUE", "debug")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_DIR = f"/home/{ALCF_USERNAME}"
JOB_NAME = f"amsc-rig-alcf-{RUN_ID}"

if SUBMIT_JOB:
    missing = [name for name, value in {"ALCF_USERNAME": ALCF_USERNAME, "ALCF_ACCOUNT": ALCF_ACCOUNT, "ALCF_QUEUE": ALCF_QUEUE}.items() if not value]
    if missing:
        raise RuntimeError("Missing required environment settings: " + ", ".join(missing))
    stdout_path = f"{OUTPUT_DIR}/{JOB_NAME}.stdout"
    stderr_path = f"{OUTPUT_DIR}/{JOB_NAME}.stderr"
    job = polaris.submit(
        executable="/bin/echo", arguments=["Hello from AmSC through RIG", RUN_ID],
        directory=OUTPUT_DIR, name=JOB_NAME, queue=ALCF_QUEUE, account=ALCF_ACCOUNT,
        duration=300, nodes=1, stdout_path=stdout_path, stderr_path=stderr_path,
        filesystems="home",
    )
else:
    print("SUBMIT_JOB=False: no scheduler job was submitted.")

## Bounded wait and output read
This executes only if the default-off submission gate was enabled and submission succeeded.

In [ ]:
if SUBMIT_JOB:
    completed = job.wait(timeout=600, poll_interval=10)
    output_task = home.fs.head(stdout_path, lines=50)
    print(output_task.result)